In [1]:
import plotly.graph_objects as go
from pathlib import Path

In [2]:
transactions = [
    # origin entity, ordering bank, intermediary region, jurisdiction, recipient cluster, value
    ("Origin Entity A", "Ordering Bank 1", "Region Alpha", "Jurisdiction A1", "Recipient Cluster 1", 42),
    ("Origin Entity A", "Ordering Bank 1", "Region Alpha", "Jurisdiction A2", "Recipient Cluster 2", 18),
    ("Origin Entity A", "Ordering Bank 2", "Region Alpha", "Jurisdiction A1", "Recipient Cluster 1", 22),
    ("Origin Entity A", "Ordering Bank 2", "Region Beta", "Jurisdiction B1", "Recipient Cluster 3", 6),
    ("Origin Entity A", "Ordering Bank 2", "Region Gamma", "Jurisdiction C1", "Recipient Cluster 4", 5),

    ("Origin Entity B", "Ordering Bank 3", "Region Delta", "Jurisdiction D1", "Recipient Cluster 2", 24),
    ("Origin Entity B", "Ordering Bank 3", "Region Alpha", "Jurisdiction A1", "Recipient Cluster 1", 20),
    ("Origin Entity B", "Ordering Bank 3", "Region Alpha", "Jurisdiction A3", "Recipient Cluster 5", 7),
    ("Origin Entity B", "Ordering Bank 4", "Region Delta", "Jurisdiction D2", "Recipient Cluster 4", 8),
    ("Origin Entity B", "Ordering Bank 4", "Region Beta", "Jurisdiction B2", "Recipient Cluster 2", 4),

    ("Origin Entity C", "Ordering Bank 3", "Region Alpha", "Jurisdiction A1", "Recipient Cluster 1", 14),
    ("Origin Entity C", "Ordering Bank 3", "Region Epsilon", "Jurisdiction E1", "Recipient Cluster 2", 6),
    ("Origin Entity C", "Ordering Bank 4", "Region Delta", "Jurisdiction D1", "Recipient Cluster 2", 10),
    ("Origin Entity C", "Ordering Bank 4", "Region Gamma", "Jurisdiction C2", "Recipient Cluster 6", 4),

    ("Origin Entity D", "Ordering Bank 5", "Region Gamma", "Jurisdiction C1", "Recipient Cluster 4", 9),
    ("Origin Entity D", "Ordering Bank 5", "Region Alpha", "Jurisdiction A1", "Recipient Cluster 1", 11),
    ("Origin Entity D", "Ordering Bank 5", "Region Beta", "Jurisdiction B3", "Recipient Cluster 3", 3),

    ("Origin Entity E", "Ordering Bank 6", "Region Beta", "Jurisdiction B1", "Recipient Cluster 3", 10),
    ("Origin Entity E", "Ordering Bank 6", "Region Beta", "Jurisdiction B2", "Recipient Cluster 2", 6),
    ("Origin Entity E", "Ordering Bank 6", "Region Zeta", "Jurisdiction Z1", "Recipient Cluster 6", 5),

    ("Origin Entity F", "Ordering Bank 2", "Region Alpha", "Jurisdiction A4", "Recipient Cluster 5", 5),
    ("Origin Entity F", "Ordering Bank 2", "Region Epsilon", "Jurisdiction E2", "Recipient Cluster 1", 4),
    ("Origin Entity F", "Ordering Bank 1", "Region Other", "Jurisdiction O1", "Other recipients", 3),
]

In [3]:
layers = {
    "origin": [
        "Origin Entity A",
        "Origin Entity B",
        "Origin Entity C",
        "Origin Entity D",
        "Origin Entity E",
        "Origin Entity F",
    ],
    "bank": [
        "Ordering Bank 1",
        "Ordering Bank 2",
        "Ordering Bank 3",
        "Ordering Bank 4",
        "Ordering Bank 5",
        "Ordering Bank 6",
    ],
    "region": [
        "Region Alpha",
        "Region Beta",
        "Region Gamma",
        "Region Delta",
        "Region Epsilon",
        "Region Zeta",
        "Region Other",
    ],
    "jurisdiction": [
        "Jurisdiction A1",
        "Jurisdiction A2",
        "Jurisdiction A3",
        "Jurisdiction A4",
        "Jurisdiction B1",
        "Jurisdiction B2",
        "Jurisdiction B3",
        "Jurisdiction C1",
        "Jurisdiction C2",
        "Jurisdiction D1",
        "Jurisdiction D2",
        "Jurisdiction E1",
        "Jurisdiction E2",
        "Jurisdiction Z1",
        "Jurisdiction O1",
    ],
    "recipient": [
        "Recipient Cluster 1",
        "Recipient Cluster 2",
        "Recipient Cluster 3",
        "Recipient Cluster 4",
        "Recipient Cluster 5",
        "Recipient Cluster 6",
        "Other recipients",
    ],
}

In [4]:
region_colors = {
    "Region Alpha": "#ef626c",
    "Region Beta": "#ff9fb2",
    "Region Gamma": "#f2c94c",
    "Region Delta": "#59b3b3",
    "Region Epsilon": "#a56f4f",
    "Region Zeta": "#9b6aa6",
    "Region Other": "#66a866",
}

layer_colors = {
    "origin": "#4c78a8",
    "bank": "#3f7cac",
    "jurisdiction": "#9aa4b2",
    "recipient": "#d1495b",
}


def hex_to_rgba(hex_color: str, alpha: float = 0.35) -> str:
    hex_color = hex_color.lstrip("#")
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return f"rgba({r}, {g}, {b}, {alpha})"

In [5]:
x_positions = {
    "origin": 0.02,
    "bank": 0.28,
    "region": 0.50,
    "jurisdiction": 0.68,
    "recipient": 0.94,
}

node_ids = []
node_labels = []
node_colors = []
node_x = []
node_y = []

for layer_name, layer_values in layers.items():
    count = len(layer_values)

    for index, label in enumerate(layer_values):
        node_ids.append((layer_name, label))
        node_labels.append(label)
        node_x.append(x_positions[layer_name])

        if count == 1:
            node_y.append(0.5)
        else:
            node_y.append(0.04 + index * (0.90 / (count - 1)))

        if layer_name == "region":
            node_colors.append(region_colors.get(label, "#999999"))
        else:
            node_colors.append(layer_colors.get(layer_name, "#999999"))

node_to_index = {node_id: index for index, node_id in enumerate(node_ids)}

In [6]:
sources = []
targets = []
values = []
link_colors = []

for origin, bank, region, jurisdiction, recipient, value in transactions:
    path = [
        ("origin", origin),
        ("bank", bank),
        ("region", region),
        ("jurisdiction", jurisdiction),
        ("recipient", recipient),
    ]

    link_color = hex_to_rgba(region_colors.get(region, "#999999"), alpha=0.32)

    for source_node, target_node in zip(path[:-1], path[1:]):
        sources.append(node_to_index[source_node])
        targets.append(node_to_index[target_node])
        values.append(value)
        link_colors.append(link_color)

In [ ]:
fig = go.Figure(
    data=[
        go.Sankey(
            arrangement="freeform",
            node=dict(
                pad=16,
                thickness=18,
                line=dict(width=0.5, color="rgba(255,255,255,0.35)"),
                label=node_labels,
                color=node_colors,
                x=node_x,
                y=node_y,
            ),
            link=dict(
                source=sources,
                target=targets,
                value=values,
                color=link_colors,
            ),
        )
    ]
)

fig.update_layout(
    title=dict(
        text="Synthetic multi-branch Sankey test",
        x=0.02,
        xanchor="left",
        y=1.0,
        yanchor="top",
    ),
    template="plotly_dark",
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    font=dict(size=12, color="#f5f7fa"),
    width=None,
    height=780,
    autosize=True,
    margin=dict(l=20, r=20, t=150, b=20),
    annotations=[
        dict(x=0.02, y=1.08, xref="paper", yref="paper", text="<b>ORIGIN ENTITY</b>", showarrow=False, font=dict(size=12)),
        dict(x=0.28, y=1.08, xref="paper", yref="paper", text="<b>ORDERING BANK</b>", showarrow=False, font=dict(size=12)),
        dict(x=0.50, y=1.08, xref="paper", yref="paper", text="<b>REGION</b>", showarrow=False, font=dict(size=12)),
        dict(x=0.68, y=1.08, xref="paper", yref="paper", text="<b>JURISDICTION</b>", showarrow=False, font=dict(size=12)),
        dict(x=0.94, y=1.08, xref="paper", yref="paper", text="<b>RECIPIENT</b>", showarrow=False, font=dict(size=12)),
    ],
)

fig.show()

In [ ]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

output_dir = project_root / "output"
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "20261003_sankey_test.html"

plot_html = fig.to_html(
    include_plotlyjs="cdn",
    full_html=False,
    config={
        "responsive": True,
        "displaylogo": False,
    },
)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Sankey | phey.app</title>
  <style>
    html,
    body {{
      margin: 0;
      padding: 0;
      min-height: 100%;
      background: #0d1117;
      color: #f5f7fa;
      font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    }}

    .plot-page {{
      min-height: 100vh;
      background: #0d1117;
      padding: clamp(0.75rem, 2vw, 1.5rem);
      box-sizing: border-box;
    }}

    .plot-wrap {{
      width: 100%;
      margin: 0 auto;
    }}

    a.back-link {{
      display: inline-flex;
      margin-bottom: 1rem;
      color: #f5f7fa;
      text-decoration: none;
      opacity: 0.75;
      font-size: 0.95rem;
    }}

    a.back-link:hover {{
      opacity: 1;
    }}
  </style>
</head>
<body>
  <main class="plot-page">
    <div class="plot-wrap">
      <a class="back-link" href="../index.html">← Back to phey.app</a>
      {plot_html}
    </div>
  </main>
</body>
</html>
"""

output_file.write_text(html, encoding="utf-8")

print(f"Exported Sankey to: {output_file.resolve()}")

Exported Sankey to: C:\Users\PH\Python\p00\output\20261003_sankey_test.html
